# D1 — Where does the radiation contrast appear?

Compare **LCC − LCC_fBVOC**, separately in PD and FUT. This notebook asks whether that contrast is confined to the modified land-cover cells or also appears in the tropics and southern extratropics.

**What this can tell us:** the size, location and year-to-year consistency of the radiation difference. **What it cannot identify alone:** how much is caused by omitted emissions, transported effects, background environmental change or other differences between the simulations.

**Set-up:** in fBVOC, MEGAN is off and only ISOP and MTERP are prescribed, from the CTRL_PD half-hourly climatology. The FUT files are the same PD climatology re-stamped to 2100. So in FUT the contrast also contains the change from PD to FUT background emissions.

**Use:** Python 3.10+; numpy, pandas, xarray, scipy, matplotlib ≥3.7, dask, cftime and a NetCDF backend. Run top to bottom. Paths, case names and spin-up removal are set in step 1 (use 0 if the inputs already exclude spin-up). Everything else is defined in this notebook; no `boreal_forest_expansion` import is needed.


In [ ]:
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass
from typing import Iterable
import glob, logging, math, re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import MaxNLocator
from scipy import stats as sps
from IPython.display import display
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("diagnostics")
pd.set_option("display.max_columns", 30, "display.width", 160)
plt.rcParams.update({"font.size": 10, "figure.dpi": 110})

@dataclass
class Contrast:
    """A difference between two members: `minuend - subtrahend` (file globs)."""
    name: str            # e.g. "BVOC_PD"
    period: str          # "PD" or "FUT"
    minuend: str         # glob of h0 files, e.g. LCC_PD
    subtrahend: str      # glob of h0 files, e.g. LCC_PD_fBVOCctrlPD_IM
    label: str = ""      # e.g. "LCC_PD - LCC_PD_fBVOCctrlPD_IM"


## 1. Set input files and output folders


In [ ]:
ARCHIVE = Path("/cluster/work/users/adelez/archive")
CASES = {   # production cases (CLAUDE.md § Simulations)
    "LCC_PD":        "NF2000norbc_tropstratchem_nudg_lcc_f19_f19-20261005",
    "LCC_PD_fBVOCctrlPD_IM":  "NF2000norbc_tropstratchem_nudg_lcc_fBVOC_f19_f19-20261008",
    "LCC_FUT":       "NF2100ssp585norbc_tropstratchem_nudg_lcc_f19_f19-20261005",
    "LCC_FUT_fBVOCctrlPD_IM": "NF2100ssp585norbc_tropstratchem_nudg_lcc_fBVOC_f19_f19-20261009",
}
PATTERNS = {alias: str(ARCHIVE / case / "atm" / "hist" / f"{case}.cam.h0.*.nc") for alias, case in CASES.items()}

REPO = Path("/cluster/home/adelez/BOREAL-FOREST-EXPANSION")
MODIFIED_CELLS = REPO / "diagnostics/check-simulation-equilibrium/lcc-modified-cells/modified_cells.nc"
if not MODIFIED_CELLS.exists():          # the CSV in the repo has the same cells
    MODIFIED_CELLS = MODIFIED_CELLS.with_suffix(".csv")

SPINUP_MONTHS = 12        # first 12 months dropped (decision in the postprocessing work)
RATIO_THRESHOLD = 0.2     # descriptive ratio flag; not a physical threshold
FDR_ALPHA = 0.10          # Benjamini-Hochberg, Wilks (2016)
FIG_DIR = REPO / "diagnostics/other-BVOC-emission-magnitude/D1_farfield_check_outputs/figures"
TABLE_DIR = REPO / "diagnostics/other-BVOC-emission-magnitude/D1_farfield_check_outputs/tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
CONTRASTS = [
    Contrast("BVOC_PD",  "PD",  PATTERNS["LCC_PD"],  PATTERNS["LCC_PD_fBVOCctrlPD_IM"],  "LCC_PD − LCC_PD_fBVOCctrlPD_IM"),
    Contrast("BVOC_FUT", "FUT", PATTERNS["LCC_FUT"], PATTERNS["LCC_FUT_fBVOCctrlPD_IM"], "LCC_FUT − LCC_FUT_fBVOCctrlPD_IM"),
]
PATTERNS


## Notebook-local helpers
These definitions live here so the notebook needs no project imports or generator scripts. The actual workflow follows below in numbered steps. Ordinary Python functions keep repeated calculations consistent; none need to be installed in `src`.


### Read monthly files and place them in the correct month


In [ ]:
log = logging.getLogger(__name__)


WEIGHT_VARS = ("gw", "GRIDAREA", "LANDFRAC", "area", "landfrac")


COMPONENT_TAGS = (".cam.", ".clm2.")


def _case_of(path: str) -> str:
    """Case name = file name up to '.cam.' or '.clm2.' (e.g. NF2000..._ctrl_f19_f19-20260925)."""
    name = Path(path).name
    for tag in COMPONENT_TAGS:
        if tag in name:
            return name.split(tag)[0]
    return name


def find_files(pattern: str | Path) -> list[str]:
    """Expand a glob and refuse to mix two different cases.

    A wildcard in the case date (e.g. '...lcc_f19_f19-*') is convenient, but if two
    runs with different dates exist the files would be silently merged. That is an
    error here, not a warning.
    """
    files = sorted(glob.glob(str(pattern)))
    if not files:
        raise FileNotFoundError(f"No files match {pattern}")
    cases = sorted({_case_of(f) for f in files})
    if len(cases) > 1:
        raise ValueError(
            f"Pattern {pattern} matches files from {len(cases)} cases: {cases}. "
            "Put the full case name (with date) in the pattern."
        )
    return files


def _bounds_name(ds: xr.Dataset) -> str | None:
    for name in ("time_bnds", "time_bounds"):
        if name in ds.variables:
            return name
    return None


def fix_time(ds: xr.Dataset) -> xr.Dataset:
    """Stamp each monthly mean at the midpoint of its averaging interval.

    CAM writes monthly means with time = end of the interval, i.e. the January mean
    is stamped 1 February (a: verified in the earlier postprocessing work). Grouping
    by year/month on the raw stamp therefore shifts everything by one month.
    Also stores the interval length in days as `days_in_interval` (used as weight).
    """
    bname = _bounds_name(ds)
    if bname is None:
        raise ValueError(
            "No time bounds in the file: cannot place monthly means correctly. "
            "Open raw h0 files, or keep time_bnds when selecting variables."
        )
    bnds = ds[bname].load()
    bdim = [d for d in bnds.dims if d != "time"][0]
    start, end = bnds.isel({bdim: 0}).values, bnds.isel({bdim: 1}).values
    length = end - start                      # timedelta64 or datetime.timedelta objects
    mid = start + length / 2
    if np.issubdtype(np.asarray(length).dtype, np.timedelta64):
        days = length / np.timedelta64(1, "D")
    else:
        days = np.array([t.total_seconds() / 86400.0 for t in length])
    out = ds.assign_coords(time=("time", mid, ds.time.attrs))
    out["days_in_interval"] = ("time", np.asarray(days, dtype=float))
    return out


def drop_spinup(ds: xr.Dataset, months: int = 12) -> xr.Dataset:
    """Remove the first `months` monthly means (decision: 12 months of spin-up)."""
    if months <= 0:
        return ds
    if ds.sizes["time"] <= months:
        raise ValueError(f"Only {ds.sizes['time']} months in file, cannot drop {months} of spin-up")
    return ds.isel(time=slice(months, None))


def open_cam_monthly(
    pattern: str | Path,
    variables: Iterable[str],
    spinup_months: int = 12,
    strict: bool = True,
) -> xr.Dataset:
    """Open monthly CAM or CLM output for one case: selected variables, fixed time, no spin-up.

    Works for CLM h0 too (<case>.clm2.h0.*.nc): time bounds are found under either
    name, and `area`/`landfrac` are kept for land weights.

    Parameters
    ----------
    pattern
        Glob matching the h0 files of ONE case (see `find_files`).
    variables
        Data variables to read.
    spinup_months
        Number of leading months to drop.
    strict
        True: a missing variable raises KeyError. False: missing variables are listed in
        ds.attrs['missing'] and skipped.
    """
    files = find_files(pattern)
    variables = list(variables)
    keep = set(variables) | set(WEIGHT_VARS) | {"time_bnds", "time_bounds"}

    def _select(ds: xr.Dataset) -> xr.Dataset:
        absent = set(variables) - set(ds.data_vars)
        if absent and strict:
            raise KeyError(f"Missing fields in {ds.encoding.get('source', 'input')}: {sorted(absent)}")
        present = [v for v in ds.data_vars if v in keep]
        return ds[present]

    try:  # xarray >= 2025.01
        time_kw = {"decode_times": xr.coders.CFDatetimeCoder(use_cftime=True)}
    except AttributeError:  # older xarray
        time_kw = {"use_cftime": True}
    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        preprocess=_select,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        parallel=False,
        **time_kw,
    )
    missing = [v for v in variables if v not in ds.data_vars]
    if missing and strict:
        raise KeyError(f"{_case_of(files[0])}: missing {missing}. Present: {sorted(ds.data_vars)}")
    if missing:
        log.warning("%s: %d requested variables missing: %s", _case_of(files[0]), len(missing), missing)

    ds = fix_time(ds)
    keys = ds.time.dt.year.values * 12 + ds.time.dt.month.values
    if np.any(np.diff(keys) != 1):
        raise ValueError("Duplicate or missing months; check overlapping raw/prepared files.")
    # CLM's first month starts one model step (30 min) early, so allow ~1 h of slack.
    if not np.allclose(ds.days_in_interval, ds.time.dt.days_in_month, rtol=0, atol=0.05):
        raise ValueError("Intervals are not full monthly means.")
    ds = drop_spinup(ds, spinup_months)
    ds.attrs["missing"] = ",".join(missing)
    ds.attrs["case"] = _case_of(files[0])
    ds.attrs["n_files"] = len(files)
    log.info("%s: %d files, %d months after spin-up", ds.attrs["case"], len(files), ds.sizes["time"])
    return ds


def annual_mean(ds: xr.Dataset | xr.DataArray, days: xr.DataArray) -> xr.Dataset | xr.DataArray:
    """Annual means weighted by month length; incomplete years are dropped (with a log line).

    Every variable in `ds` must have a time dimension (pass only the fields to average,
    not gw or other static fields, which would otherwise be broadcast over time).
    """
    if isinstance(ds, xr.Dataset):
        static = [v for v in ds.data_vars if "time" not in ds[v].dims]
        if static:
            raise ValueError(f"annual_mean: variables without time dimension {static}; drop them first")
    year = ds["time"].dt.year
    counts = days.groupby(year).count()
    complete = counts["year"].where(counts == 12, drop=True).values
    incomplete = sorted(set(counts["year"].values) - set(complete))
    if incomplete:
        log.warning("Dropping incomplete years %s", incomplete)
    w = days.groupby(year) / days.groupby(year).sum()
    out = (ds * w).groupby(year).sum("time", skipna=False)
    return out.sel(year=complete)


### Select regions and calculate area weights


In [ ]:
log = logging.getLogger(__name__)


def area_weights(ds: xr.Dataset) -> xr.DataArray:
    """2-D (lat, lon) weights proportional to cell area, normalised to sum 1."""
    if "gw" in ds:
        w = ds["gw"]
        if "time" in w.dims:
            w = w.isel(time=0, drop=True)
        w = w.broadcast_like(ds["lon"]).transpose("lat", "lon")
        source = "gw"
    elif "GRIDAREA" in ds:
        w = ds["GRIDAREA"]
        if "time" in w.dims:
            w = w.isel(time=0, drop=True)
        source = "GRIDAREA"
    else:
        w = np.cos(np.deg2rad(ds["lat"])).broadcast_like(ds["lon"]).transpose("lat", "lon")
        source = "cos(lat)"
    w = (w / w.sum()).load()
    w.name = "area_weight"
    w.attrs["source"] = source
    return w


def band_mask(ds: xr.Dataset, lat_min: float, lat_max: float) -> xr.DataArray:
    """True for cells whose centre lies in [lat_min, lat_max]."""
    lat2d = ds["lat"].broadcast_like(ds["lon"]).transpose("lat", "lon")
    return ((lat2d >= lat_min) & (lat2d <= lat_max)).rename("mask")


def load_modified_cells(path: str | Path, ds: xr.Dataset, tol_deg: float = 1e-3) -> xr.DataArray:
    """Mask of land-cover-modified cells on the grid of `ds`.

    Accepts the cache written by build_modified_cells (modified_cells.nc, variable
    `modified`) or its CSV twin (columns lat, lon_360). Cells are matched by
    coordinate within `tol_deg`; any unmatched cell raises (never a nearest-neighbour
    regrid, consistent with equilibrium.modified_mask).
    """
    path = Path(path)
    if path.suffix == ".nc":
        with xr.open_dataset(path) as m:
            m = m.load()
        lat2d, lon2d = xr.broadcast(m["lat"], m["lon"])
        sel = m["modified"].fillna(0).transpose("lat", "lon").values.astype(bool)
        lats, lons = lat2d.values[sel], lon2d.values[sel] % 360
    elif path.suffix == ".csv":
        t = pd.read_csv(path)
        lats, lons = t["lat"].to_numpy(), t["lon_360"].to_numpy() % 360
    else:
        raise ValueError(f"Expected .nc or .csv, got {path}")

    if len(lats) == 0:
        raise ValueError("The modified-cell mask is empty.")
    glat, glon = ds["lat"].values, ds["lon"].values % 360
    i = np.abs(glat[None, :] - lats[:, None]).argmin(axis=1)
    dlon = np.abs(glon[None, :] - lons[:, None])
    dlon = np.minimum(dlon, 360 - dlon)
    j = dlon.argmin(axis=1)
    err = np.maximum(np.abs(glat[i] - lats), dlon[np.arange(len(j)), j])
    if np.any(err > tol_deg):
        raise ValueError(f"{(err > tol_deg).sum()} modified cells do not match the grid within {tol_deg} deg")

    mask = np.zeros((glat.size, glon.size), dtype=bool)
    mask[i, j] = True
    log.info("Modified cells: %d from %s", mask.sum(), path.name)
    return xr.DataArray(mask, coords={"lat": ds["lat"], "lon": ds["lon"]}, dims=("lat", "lon"), name="modified")


def default_regions(ds: xr.Dataset, modified: xr.DataArray) -> dict[str, xr.DataArray]:
    """Regions for the far-field check.

    tropics, SH_extratropics: far from the land-cover change (all modified cells are
    north of 46 N (a, modified_cells.csv)); modified_cells: where the BVOC signal is
    expected, shown for scale; global: everything.
    """
    regions = {
        "modified_cells": modified,
        "tropics": band_mask(ds, -30, 30),
        "SH_extratropics": band_mask(ds, -90, -30.0001),
        "global": xr.ones_like(modified, dtype=bool),
    }
    for name in ("tropics", "SH_extratropics"):
        if bool((regions[name] & modified).any()):
            raise ValueError(f"Region {name} overlaps modified cells: not a far-field region")
    return regions


def area_fraction(weights: xr.DataArray, mask: xr.DataArray) -> float:
    return float(weights.where(mask).sum())


def regional_mean(da: xr.DataArray, weights: xr.DataArray, mask: xr.DataArray) -> xr.DataArray:
    """Area-weighted mean over the cells where mask is True (keeps other dims, e.g. year)."""
    w = weights.where(mask, 0.0)
    return da.weighted(w).mean(("lat", "lon"))


### Summarise paired annual differences


In [ ]:
def lag1_autocorr(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 3 or np.ptp(x) == 0:
        return np.nan
    a = x - x.mean()
    return float(np.sum(a[1:] * a[:-1]) / np.sum(a * a))


def paired_summary(d: xr.DataArray | np.ndarray, conf: float = 0.95, n_boot: int = 10000,
                   seed: int = 0) -> dict:
    """Mean of yearly differences with t-based and bootstrap intervals.

    Returns mean, sd, n, r1, n_eff, se (with n_eff), ci_lo/ci_hi (t, n_eff - 1 dof),
    p (two-sided, H0: mean = 0), boot_lo/boot_hi (percentile bootstrap over years;
    with only ~9 years it is narrower than the t interval and is shown for reference).
    """
    x = np.asarray(d, dtype=float).ravel()
    x = x[np.isfinite(x)]
    n = x.size
    out = {"mean": np.nan, "sd": np.nan, "n": n, "r1": np.nan, "n_eff": np.nan, "se": np.nan,
           "ci_lo": np.nan, "ci_hi": np.nan, "p": np.nan, "boot_lo": np.nan, "boot_hi": np.nan}
    if n < 3:
        return out
    mean, sd = x.mean(), x.std(ddof=1)
    r1 = lag1_autocorr(x)
    r1_pos = max(r1, 0.0) if np.isfinite(r1) else 0.0
    n_eff = max(n * (1 - r1_pos) / (1 + r1_pos), 2.0)
    se = sd / np.sqrt(n_eff)
    tcrit = sps.t.ppf(0.5 + conf / 2, df=n_eff - 1)
    p = 2 * sps.t.sf(abs(mean) / se, df=n_eff - 1) if se > 0 else (0.0 if mean != 0 else 1.0)
    rng = np.random.default_rng(seed)
    boot = x[rng.integers(0, n, size=(n_boot, n))].mean(axis=1)
    lo, hi = np.percentile(boot, [50 * (1 - conf), 50 * (1 + conf)])
    out.update(mean=mean, sd=sd, r1=r1, n_eff=n_eff, se=se, ci_lo=mean - tcrit * se,
               ci_hi=mean + tcrit * se, p=p, boot_lo=lo, boot_hi=hi)
    return out


def cellwise_pvalue(d: xr.DataArray, dim: str = "year") -> xr.DataArray:
    """Two-sided one-sample t-test of the yearly differences in every grid cell."""
    def _p(x):
        with np.errstate(invalid="ignore", divide="ignore"):
            return sps.ttest_1samp(x, 0.0, axis=-1, nan_policy="omit").pvalue
    p = xr.apply_ufunc(_p, d, input_core_dims=[[dim]], dask="allowed")
    return p.rename("p")


def fdr_mask(p: xr.DataArray, alpha: float = 0.10, within: xr.DataArray | None = None) -> xr.DataArray:
    """Benjamini-Hochberg false discovery rate control (Wilks 2016 suggests alpha_FDR = 0.10).

    Cells with p_(i) <= (i / N) * alpha are significant, where N counts the cells
    tested. `within` restricts the test to a region (e.g. 54-90 N for the maps), so N
    is the number of cells in that region.
    """
    pv = p.values.copy()
    valid = np.isfinite(pv)
    if within is not None:
        valid &= within.values.astype(bool)
    flat = pv[valid]
    sig = np.zeros_like(pv, dtype=bool)
    if flat.size:
        order = np.argsort(flat)
        ranked = flat[order]
        thresh = alpha * np.arange(1, flat.size + 1) / flat.size
        below = np.nonzero(ranked <= thresh)[0]
        if below.size:
            cutoff = ranked[below.max()]
            sig[valid] = flat <= cutoff
    return xr.DataArray(sig, coords=p.coords, dims=p.dims, name="fdr_significant")


### Split radiation into direct aerosol, cloud and clean clear-sky terms


In [ ]:
GHAN_INPUTS = ["FSNT", "FLNT", "FSNT_DRF", "FLNT_DRF", "FSNTCDRF", "FLNTCDRF"]


COMPONENTS = ["ari", "cloud", "clear", "total"]


BANDS = ["SW", "LW", "net"]


COMPONENT_LABELS = {
    "ari": "ari (direct)",
    "cloud": "cloud",
    "clear": "clean clear-sky",
    "total": "total",
}


def term_name(component: str, band: str) -> str:
    return f"{component}_{band}"


def ghan_terms(ds: xr.Dataset) -> xr.Dataset:
    """Return the 12 terms (4 components x SW/LW/net), downward positive, W m-2."""
    missing = [v for v in GHAN_INPUTS if v not in ds]
    if missing:
        raise KeyError(f"Ghan decomposition needs {missing}")

    sw = {
        "ari": ds.FSNT - ds.FSNT_DRF,
        "cloud": ds.FSNT_DRF - ds.FSNTCDRF,
        "clear": ds.FSNTCDRF,
        "total": ds.FSNT,
    }
    lw = {  # minus sign: FLNT is upward positive
        "ari": -(ds.FLNT - ds.FLNT_DRF),
        "cloud": -(ds.FLNT_DRF - ds.FLNTCDRF),
        "clear": -ds.FLNTCDRF,
        "total": -ds.FLNT,
    }
    out = xr.Dataset()
    for c in COMPONENTS:
        out[term_name(c, "SW")] = sw[c]
        out[term_name(c, "LW")] = lw[c]
        out[term_name(c, "net")] = sw[c] + lw[c]
        for b in BANDS:
            out[term_name(c, b)].attrs = {
                "units": "W m-2",
                "long_name": f"{COMPONENT_LABELS[c]} {b}, TOA, downward positive (Ghan 2013)",
            }
    return out


### Build regional tables
The ratio compares two parts of the same contrast. Neither region is assumed to contain a pure causal effect.


In [ ]:


def _same_grid(a: xr.Dataset, b: xr.Dataset) -> None:
    for c in ("lat", "lon"):
        if a.sizes[c] != b.sizes[c] or not np.allclose(a[c].values, b[c].values, atol=1e-6):
            raise ValueError(f"{a.attrs.get('case')} and {b.attrs.get('case')} differ in {c}")


def regional_table(diffs: dict[str, xr.Dataset], contrasts: list[Contrast], weights: xr.DataArray,
                   regions: dict[str, xr.DataArray]) -> pd.DataFrame:
    """Paired statistics of regional means for every contrast, term and region."""
    rows = []
    period = {c.name: c.period for c in contrasts}
    for cname, diff in diffs.items():
        for comp in COMPONENTS:
            for band in BANDS:
                term = term_name(comp, band)
                for rname, mask in regions.items():
                    series = regional_mean(diff[term], weights, mask)
                    s = paired_summary(series.values)
                    frac = area_fraction(weights, mask)
                    rows.append({
                        "contrast": cname, "period": period[cname], "component": comp, "band": band,
                        "term": term, "region": rname, "area_frac": frac, **s,
                        "contrib": s["mean"] * frac, "contrib_lo": s["ci_lo"] * frac,
                        "contrib_hi": s["ci_hi"] * frac,
                        "years": ",".join(str(int(y)) for y in diff["year"].values),
                    })
    return pd.DataFrame(rows)


def assess(table: pd.DataFrame, far_region: str = "tropics", scale_region: str = "modified_cells",
           ratio_threshold: float = 0.2) -> pd.DataFrame:
    """One verdict per contrast and term.

    detected : the far-field 95% interval excludes zero.
    ratio    : |far-field contribution to the global mean| / |modified-cells contribution|.
    verdict  :
      'none detected'           - far field not distinguishable from zero
      'detected, small'         - detected, ratio < ratio_threshold
      'detected, MATTERS'       - detected, ratio >= ratio_threshold
      'detected, no LCC signal' - detected but the modified-cells signal itself is not
                                  significant, so the ratio is not meaningful
    The threshold is a descriptive screening convention, not a causal test.
    """
    rows = []
    for (cname, term), g in table.groupby(["contrast", "term"], sort=False):
        far = g[g.region == far_region].iloc[0]
        loc = g[g.region == scale_region].iloc[0]
        detected = bool(far.ci_lo > 0 or far.ci_hi < 0)
        loc_sig = bool(loc.ci_lo > 0 or loc.ci_hi < 0)
        ratio = abs(far.contrib) / abs(loc.contrib) if loc.contrib != 0 else np.inf
        if not detected:
            verdict = "none detected"
        elif not loc_sig:
            verdict = "detected, no LCC signal"
        elif ratio >= ratio_threshold:
            verdict = "detected, MATTERS"
        else:
            verdict = "detected, small"
        rows.append({
            "contrast": cname, "period": far.period, "component": far.component, "band": far.band,
            "term": term, f"{far_region}_mean": far["mean"], f"{far_region}_ci_lo": far.ci_lo,
            f"{far_region}_ci_hi": far.ci_hi, f"{far_region}_p": far.p,
            f"{far_region}_contrib": far.contrib, f"{scale_region}_contrib": loc.contrib,
            "ratio": ratio, "verdict": verdict,
        })
    return pd.DataFrame(rows)


def mean_maps(diff: xr.Dataset, terms: list[str], alpha: float = 0.10) -> tuple[xr.Dataset, xr.Dataset]:
    """Time-mean difference maps and global FDR significance masks for `terms`."""
    means = diff[terms].mean("year")
    sig = xr.Dataset({t: fdr_mask(cellwise_pvalue(diff[t]), alpha=alpha) for t in terms})
    return means, sig


### Plotting helpers
Plain latitude–longitude maps are the offline default. Cartopy is optional.


In [ ]:
HAS_CARTOPY = False  # Set True only if Cartopy AND its coastline data are installed.
if HAS_CARTOPY:
    import cartopy.crs as ccrs
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
def _coastlines_available():
    # Never trigger a network download on a compute node.
    if not HAS_CARTOPY:
        return False
    import cartopy
    from pathlib import Path
    rel = Path("shapefiles/natural_earth/physical/ne_110m_coastline.shp")
    return any((Path(cartopy.config[k]) / rel).is_file() for k in ("data_dir", "pre_existing_data_dir"))



REGION_STYLE = {
    "modified_cells": {"color": "#2a78d6", "label": "modified cells"},
    "tropics": {"color": "#eb6834", "label": "30S-30N (far field)"},
    "SH_extratropics": {"color": "#1baf7a", "label": "90S-30S (far field)"},
}


PERIOD_STYLE = {"PD": {"color": "#2a78d6"}, "FUT": {"color": "#eb6834"}}


INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"


BAND_LABEL = {"SW": "shortwave", "LW": "longwave", "net": "net (SW + LW)"}


def _style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(INK2)
    ax.tick_params(colors=INK2, labelsize=8)
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    ax.axhline(0, color=INK2, lw=0.8)


def plot_contributions(table: pd.DataFrame, regions=("modified_cells", "tropics", "SH_extratropics"),
                       periods=("PD", "FUT"), bands=("SW", "LW", "net")) -> plt.Figure:
    """Contribution to the global mean (regional mean x area fraction), 95% intervals.

    Same y-scale within a row so PD and FUT can be compared directly.
    """
    fig, axes = plt.subplots(len(bands), len(periods), figsize=(4.2 * len(periods), 2.6 * len(bands)),
                             sharey="row", squeeze=False, constrained_layout=True)
    x = np.arange(len(COMPONENTS))
    width = 0.8 / len(regions)
    for i, band in enumerate(bands):
        for j, period in enumerate(periods):
            ax = axes[i, j]
            sub = table[(table.period == period) & (table.band == band)]
            for k, r in enumerate(regions):
                rs = sub[sub.region == r].set_index("component").reindex(COMPONENTS)
                xs = x - 0.4 + width * (k + 0.5)
                ax.bar(xs, rs["contrib"], width * 0.9, color=REGION_STYLE[r]["color"],
                       label=REGION_STYLE[r]["label"], zorder=2)
                err = np.vstack([rs["contrib"] - rs["contrib_lo"], rs["contrib_hi"] - rs["contrib"]])
                ax.errorbar(xs, rs["contrib"], yerr=err, fmt="none", ecolor=INK2, elinewidth=0.9,
                            capsize=2, zorder=3)
            _style(ax)
            ax.set_xticks(x, [COMPONENT_LABELS[c] for c in COMPONENTS], fontsize=8, color=INK)
            if i == 0:
                label = sub["contrast"].iloc[0] if len(sub) else period
                ax.set_title(f"{period}: {label}", fontsize=10, color=INK)
            if j == 0:
                ax.set_ylabel(f"{BAND_LABEL[band]}\nW m$^{{-2}}$ of global mean", fontsize=8, color=INK2)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper center", ncol=len(regions), frameon=False, fontsize=8)
    fig.suptitle("BVOC contrast: contribution to the global-mean $\\Delta$F, modified region vs far field "
                 "(bars: mean, whiskers: 95% interval over years)", fontsize=9, color=INK2, y=-0.02)
    return fig


_COAST_OK = None


def _map_axes(fig, nrows, ncols):
    axes = np.empty((nrows, ncols), dtype=object)
    for i in range(nrows):
        for j in range(ncols):
            kw = {"projection": ccrs.Robinson()} if HAS_CARTOPY else {}
            axes[i, j] = fig.add_subplot(nrows, ncols, i * ncols + j + 1, **kw)
    return axes


def plot_maps(means: dict[str, xr.Dataset], sigs: dict[str, xr.Dataset], terms: list[str],
              modified: xr.DataArray | None = None, vmax_quantile: float = 0.98) -> plt.Figure:
    """Rows = terms, columns = contrasts. Diverging colour scale centred on 0, shared per row.

    The colour range is set from cells OUTSIDE the modified region (the far field is
    what D1 is about); the stronger local signal saturates (arrow ends of the colour bar).
    Dots: FDR-significant cells (alpha 0.10, global test; expected false-discovery fraction is controlled under the test assumptions). Dashed lines: 30S and 30N. Black contour:
    modified cells.
    """
    names = list(means)
    fig = plt.figure(figsize=(4.6 * len(names) + 0.8, 2.5 * len(terms)), constrained_layout=True)
    axes = _map_axes(fig, len(terms), len(names))
    tr = {"transform": ccrs.PlateCarree()} if HAS_CARTOPY else {}
    outside = ~modified.values.astype(bool) if modified is not None else None
    for i, term in enumerate(terms):
        vals = np.concatenate([
            np.abs(means[n][term].values[outside] if outside is not None else means[n][term].values).ravel()
            for n in names])
        vmax = float(np.nanquantile(vals, vmax_quantile)) or 1e-3
        norm = TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)
        for j, n in enumerate(names):
            ax = axes[i, j]
            da = means[n][term]
            lon = da["lon"].values
            # close the seam at the dateline for a continuous map
            da_c = xr.concat([da, da.isel(lon=0).assign_coords(lon=lon[0] + 360)], dim="lon")
            m = ax.pcolormesh(da_c["lon"], da_c["lat"], da_c, cmap="RdBu_r", norm=norm, shading="nearest", **tr)
            sig = sigs[n][term]
            yy, xx = np.nonzero(sig.values)
            if yy.size:
                ax.scatter(lon[xx], da["lat"].values[yy], s=0.6, c=INK, linewidths=0, **tr)
            for lat0 in (-30, 30):
                ax.plot([-180, 180] if HAS_CARTOPY else [0, 360], [lat0, lat0], ls="--", lw=0.6, color=INK2, **tr)
            if modified is not None:
                ax.contour(modified["lon"], modified["lat"], modified.astype(float), levels=[0.5],
                           colors=INK, linewidths=0.6, **tr)
            if HAS_CARTOPY:
                if _coastlines_available():
                    ax.coastlines(lw=0.3, color=INK2)
                ax.set_global()
            if not HAS_CARTOPY:
                ax.set(xlim=(0, 360), ylim=(-90, 90), xlabel="longitude (degrees east)", ylabel="latitude (degrees north)")
            if i == 0:
                ax.set_title(n, fontsize=10, color=INK)
        cb = fig.colorbar(m, ax=list(axes[i, :]), shrink=0.85, pad=0.01, extend="both")
        cb.set_label(f"$\\Delta$ {term}\nW m$^{{-2}}$", fontsize=8, color=INK2)
        cb.ax.tick_params(labelsize=7, colors=INK2)
    return fig


def plot_yearly(diffs: dict[str, xr.Dataset], weights: xr.DataArray, mask: xr.DataArray,
                terms=("total_net", "cloud_SW", "clear_SW", "clear_LW"), region_label="30S-30N",
                periods: dict[str, str] | None = None) -> plt.Figure:
    """Year-by-year regional means of the BVOC contrast. Shaded band: 95% interval of the mean."""

    fig, axes = plt.subplots(1, len(terms), figsize=(3.2 * len(terms), 2.6), constrained_layout=True,
                             squeeze=False)
    for ax, term in zip(axes[0], terms):
        for name, diff in diffs.items():
            period = (periods or {}).get(name, diff.attrs.get("period", "PD"))
            col = PERIOD_STYLE.get(period, {"color": INK2})["color"]
            s = regional_mean(diff[term], weights, mask)
            st = paired_summary(s.values)
            ax.plot(s["year"], s, marker="o", ms=4, lw=2, color=col, label=name)
            ax.axhspan(st["ci_lo"], st["ci_hi"], color=col, alpha=0.12, lw=0)
        _style(ax)
        ax.set_title(f"$\\Delta$ {term}, {region_label}", fontsize=9, color=INK)
        ax.set_xlabel("year", fontsize=8, color=INK2)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[0, 0].set_ylabel("W m$^{-2}$ (regional mean)", fontsize=8, color=INK2)
    axes[0, 0].legend(frameon=False, fontsize=8)
    return fig


## 2. Load and form paired annual differences
Each monthly mean is assigned to the midpoint of its time bounds. Complete years are weighted by days, then the same year is subtracted between members. Exact grids and identical retained years are required. Positive radiation differences are downward, in W m⁻².


In [ ]:
diffs, grid = {}, None
for contrast in CONTRASTS:
    a = open_cam_monthly(contrast.minuend, GHAN_INPUTS, SPINUP_MONTHS)
    b = open_cam_monthly(contrast.subtrahend, GHAN_INPUTS, SPINUP_MONTHS)
    xr.align(a.lat, b.lat, join="exact")
    xr.align(a.lon, b.lon, join="exact")
    ta = annual_mean(ghan_terms(a), a.days_in_interval)
    tb = annual_mean(ghan_terms(b), b.days_in_interval)
    ta, tb = xr.align(ta, tb, join="exact")
    if ta.sizes["year"] < 3:
        raise ValueError("Need at least three complete paired years.")
    diff = (ta - tb).load()
    diff.attrs.update(label=contrast.label, period=contrast.period, minuend=a.attrs["case"], subtrahend=b.attrs["case"])
    diffs[contrast.name] = diff
    for band in BANDS:
        xr.testing.assert_allclose(diff[f"total_{band}"], diff[f"ari_{band}"] + diff[f"cloud_{band}"] + diff[f"clear_{band}"],
                                   rtol=0, atol=1e-3)  # CAM fluxes are float32: rounding ~1e-5 W m-2
    if grid is None:
        grid = a[[v for v in WEIGHT_VARS if v in a]].isel(time=0, drop=True).load() if "time" in a[[v for v in WEIGHT_VARS if v in a]].dims else a[[v for v in WEIGHT_VARS if v in a]].load()
        grid = grid.assign_coords(lat=a.lat, lon=a.lon)
    else:
        xr.align(grid.lat, a.lat, join="exact")
        xr.align(grid.lon, a.lon, join="exact")
    print(contrast.name, diff.year.values, contrast.label)
    a.close()
    b.close()


## 3. Compare regions on the same global scale
A regional mean describes the average within that region. Multiplying by the region's fraction of Earth's area gives its contribution to the global mean. Example: 0.1 W m⁻² over half the globe contributes 0.05 W m⁻² globally.

The modified cells are a reference region, not a pure forest-emissions effect. The tropical ratio is descriptive and is undefined here when the modified-region mean is not distinguishable from zero. Confidence intervals are approximate with few years; map t-tests do not use the regional autocorrelation adjustment.


In [ ]:
weights = area_weights(grid)
modified = load_modified_cells(MODIFIED_CELLS, grid)
regions = default_regions(grid, modified)
table = regional_table(diffs, CONTRASTS, weights, regions)
assessment = assess(table, ratio_threshold=RATIO_THRESHOLD)
assessment["verdict"] = assessment.verdict.replace({
    "none detected": "not distinguishable from zero",
    "detected, small": "detected; below chosen ratio",
    "detected, MATTERS": "detected; above chosen ratio",
    "detected, no LCC signal": "detected; modified-region mean uncertain",
})
assessment.loc[assessment.verdict.eq("detected; modified-region mean uncertain"), "ratio"] = np.nan
for name, data in [("regional_stats", table), ("assessment", assessment)]:
    data.to_csv(TABLE_DIR / f"D1_farfield_{name}.csv", index=False)
display(assessment.round(4))
display(table.query("band == 'net'")[["contrast", "term", "region", "mean", "ci_lo", "ci_hi", "area_frac", "contrib"]].round(4))
print("Weights:", weights.attrs["source"])


## 4. Plot regional contributions


In [ ]:
fig = plot_contributions(table)
fig.savefig(FIG_DIR / "D1_contributions.png", dpi=200, bbox_inches="tight")
plt.show()


## 5. Map the contrast
Dots indicate cellwise tests passing Benjamini–Hochberg FDR screening at the configured level, separately for each map. They do not identify a mechanism. The colour range uses the 98th percentile outside modified cells; stronger values can saturate.


In [ ]:
TERMS = ["total_net", "cloud_SW", "clear_SW", "clear_LW"]
means, sigs = {}, {}
for name, diff in diffs.items():
    means[name], sigs[name] = mean_maps(diff, TERMS, alpha=FDR_ALPHA)
fig = plot_maps(means, sigs, TERMS, modified=modified)
fig.savefig(FIG_DIR / "D1_maps.png", dpi=200, bbox_inches="tight")
plt.show()


## 6. Inspect year-to-year behaviour


In [ ]:
fig = plot_yearly(diffs, weights, regions["tropics"], periods={c.name: c.period for c in CONTRASTS})
fig.savefig(FIG_DIR / "D1_yearly_tropics.png", dpi=200, bbox_inches="tight")
plt.show()


## 7. What to conclude after running
- A persistent far-field difference means the contrast extends beyond the modified cells. It warrants investigating the simulation setup and atmospheric pathways; it does not isolate an artifact.
- FUT and PD need separate interpretations because FUT fBVOC uses a PD emissions reference.
- A clean clear-sky LW signal is not an ozone attribution; cloud-term differences are not uniquely aerosol–cloud effects.
- Failure to detect a difference is not evidence that all omitted-species effects are negligible. The 0.2 ratio is a chosen screening level, not a rule for keeping or discarding simulations.

Record actual values, uncertainty and case configuration before deciding on new runs. Use D2 to inspect emission budgets, then check namelists, forcing and vegetation states if unexpected differences remain.
